In [ ]:
from google.colab import files
files.upload()

Saving kaggle.json to kaggle.json


{'kaggle.json': b'{"username":"seaazy","key":"8132ecfe03940a067c9f784287ec3c57"}'}

In [ ]:
import os
!mkdir -p ~/.kaggle
!cp kaggle.json ~/.kaggle/
!chmod 600 ~/.kaggle/kaggle.json

In [ ]:
!kaggle datasets download -d elvis23/mental-health-conversational-data

Dataset URL: https://www.kaggle.com/datasets/elvis23/mental-health-conversational-data
License(s): copyright-authors
  0% 0.00/11.8k [00:00<?, ?B/s]
100% 11.8k/11.8k [00:00<00:00, 41.8MB/s]


In [ ]:
!unzip mental-health-conversational-data -d ./data

Archive:  mental-health-conversational-data.zip
  inflating: ./data/intents.json     


In [ ]:
import os
import json
import random
import numpy as np
import pandas as pd

# Semantic embeddings
from sentence_transformers import SentenceTransformer, util

# Ensure reproducibility
random.seed(42)

# Load data
with open('./data/intents.json', 'r') as f:
    data = json.load(f)

data_list = []
for intent in data['intents']:
    tag = intent['tag']
    responses = intent.get('responses', [])
    for pattern in intent.get('patterns', []):
        data_list.append({'tag': tag, 'pattern': pattern, 'responses': responses})
if not data_list:
    raise ValueError("No data loaded from intents.json")

# DataFrame
df_expanded = pd.DataFrame(data_list)

# Load SentenceTransformer model
model_name = 'all-MiniLM-L6-v2'
embedder = SentenceTransformer(model_name)

# Precompute embeddings for all patterns
patterns = df_expanded['pattern'].tolist()
pattern_embeddings = embedder.encode(patterns, convert_to_tensor=True)

def predict_intent(user_input, threshold=0.6):
    # Compute embedding for user input
    user_emb = embedder.encode(user_input, convert_to_tensor=True)
    # Compute cosine similarities
    cos_scores = util.cos_sim(user_emb, pattern_embeddings)[0]
    # Find best match
    top_idx = int(np.argmax(cos_scores))
    score = float(cos_scores[top_idx])
    if score < threshold:
        return 'unknown', score
    return df_expanded.iloc[top_idx]['tag'], score


def generate_response(intent):
    if intent == 'unknown':
        return "I'm not sure I understand. Could you rephrase?"
    # Collect possible responses and choose randomly
    responses = df_expanded[df_expanded['tag'] == intent]['responses'].iloc[0]
    return random.choice(responses)

# Chat loop
if __name__ == '__main__':
    print("Start chatting with the Semantic Chatbot (type 'quit' to stop)")
    while True:
        user_input = input("You: ")
        if user_input.lower() in ['quit', 'exit']:
            print("Chatbot: Goodbye!")
            break
        intent, conf = predict_intent(user_input)
        response = generate_response(intent)
        print(f"[Intent: {intent}, Similarity: {conf:.2f}]")
        print("Chatbot:", response)


/usr/local/lib/python3.11/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Start chatting with the Semantic Chatbot (type 'quit' to stop)
[Intent: greeting, Similarity: 1.00]
Chatbot: Hello there. Tell me how are you feeling today?
[Intent: thanks, Similarity: 1.00]
Chatbot: My pleasure
[Intent: happy, Similarity: 0.95]
Chatbot: That's geat to hear. I'm glad you're feeling this way.
[Intent: death, Similarity: 1.00]
Chatbot: I'm sorry to hear that. If you want to talk about it. I'm here.
[Intent: death, Similarity: 1.00]
Chatbot: I'm sorry to hear that. If you want to talk about it. I'm here.
[Intent: death, Similarity: 0.75]
Chatbot: My condolences. I'm here if you need to talk.
[Intent: unknown, Similarity: 0.56]
Chatbot: I'm not sure I understand. Could you rephrase?
